In [0]:
# Padronização de nomenclatura e caminhos do projeto
catalog = "case"
bronze_schema = f"{catalog}.bronze"
landing_path = f"/Volumes/{catalog}/bronze/inputs"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.inputs")

print(f"Schema {bronze_schema} garantido no ambiente.")

Schema case.bronze garantido no ambiente.


In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento: "Nome_do_Arquivo.csv": "Nome_da_Tabela_Bronze"
arquivos_tabelas = {
    "chamados.csv": "tb_chamados",
    "Chamados_Hora.CSV": "tb_chamados_hora",
    "clientes.csv": "tb_clientes",
    "custos.csv": "tb_custos",
    "pesquisa_satisfacao.csv": "tb_pesquisa_satisfacao",
    "canais.csv": "tb_canais",
    "base_motivos.csv": "tb_base_motivos",
    "base_atendentes.csv": "tb_base_atendentes"
}

try:
    # Lista os arquivos fisicamente presentes no Volume
    arquivos_no_volume = [f.name for f in dbutils.fs.ls(landing_path)]
except Exception:
    arquivos_no_volume = []
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Verifique se o diretório existe.")

for arquivo, tabela in arquivos_tabelas.items():
    if arquivo in arquivos_no_volume:
        caminho = f"{landing_path}/{arquivo}"
        
        # Leitura bruta (inferSchema=True facilita a primeira carga)
        df_raw = spark.read.csv(caminho, header=True, inferSchema=True)
        
        # Adição da coluna de auditoria e salvamento
        df_raw \
            .withColumn("ingestion_datetime", current_timestamp()) \
            .write.format("delta").mode("append") \
            .saveAsTable(f"{bronze_schema}.{tabela}")

        print(f"[OK] {tabela} ingerida com sucesso ({df_raw.count()} linhas lidas do arquivo).")
    else:
        print(f"[ERRO] Arquivo '{arquivo}' não encontrado. Faça o upload para o Volume.")

# Debug
for tabela in arquivos_tabelas.values():
    print(f"\n--- {tabela} ---")
    display(spark.table(f"{bronze_schema}.{tabela}").limit(10))

[OK] tb_chamados ingerida com sucesso (134474 linhas lidas do arquivo).
[OK] tb_chamados_hora ingerida com sucesso (131838 linhas lidas do arquivo).
[OK] tb_clientes ingerida com sucesso (47705 linhas lidas do arquivo).
[OK] tb_custos ingerida com sucesso (131069 linhas lidas do arquivo).
[OK] tb_pesquisa_satisfacao ingerida com sucesso (81021 linhas lidas do arquivo).
[OK] tb_canais ingerida com sucesso (4 linhas lidas do arquivo).
[OK] tb_base_motivos ingerida com sucesso (14 linhas lidas do arquivo).
[OK] tb_base_atendentes ingerida com sucesso (25 linhas lidas do arquivo).

--- tb_chamados ---


id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente,ingestion_datetime
76696,74719850215,Renegociação de dívida,atendimento inicial,Sim,null,null,null,49,441,4,2026-10-16T21:35:13.457Z
44377,92983951302,CONSULTA DE CONTRATO,Atendimento_Inicial,Sim,null,null,null,374,355,8,2026-10-16T21:35:13.457Z
34306,77740207578,Consulta de limite,ATENDIMENTO INICIAL,Sim,null,null,null,110,985,6,2026-10-16T21:35:13.457Z
52060,36276971799,Renegociação de dívida,Ura,Não,null,igual a hora de abertura,null,NULL,216,9,2026-10-16T21:35:13.457Z
25271,53061261807,Renegociação de dívida,Atendimento_Inicial,N,null,null,null,96,384,2,2026-10-16T21:35:13.457Z
88817,28669204533,Contratação de cartão adicional,Atendimento Inicial,Não,null,null,null,344,891,23,2026-10-16T21:35:13.457Z
89333,52402176464,RENEGOCIAÇÃO DE DÍVIDA,CHATBOT,N,null,igual a hora de abertura,null,NULL,104,3,2026-10-16T21:35:13.457Z
128376,99286903466,Compra não autorizada,Chatbot,Não,null,igual a hora de abertura,null,NULL,39,25,2026-10-16T21:35:13.457Z
14867,14853867391,Dúvidas gerais sobre programa de pontos,Chatbot,Não,null,igual a hora de abertura,null,NULL,118,17,2026-10-16T21:35:13.457Z
55810,23896919281,Contestação de fatura,CHATBOT,Não,null,igual a hora de abertura,null,NULL,110,5,2026-10-16T21:35:13.457Z



--- tb_chamados_hora ---


ID_Chamado,ID_Cliente,Hora_Abertura_Chamado,Hora_Inicio_Atendimento,Hora_Finalizacao_Atendimento,ingestion_datetime
121151,25203409261,29/06/2026 �s 14:12:48,29/06/2026 �s 14:21:24,29/06/2026 �s 15:06:16,2026-10-16T21:33:39.751Z
48388,99076736475,16/09/2025 �s 09:03:42,16/09/2025 �s 09:07:41,16/09/2025 �s 10:11:50,2026-10-16T21:33:39.751Z
12605,13153799751,03/03/2025 �s 14:02:31,03/03/2025 �s 14:02:31,03/03/2025 �s 14:04:25,2026-10-16T21:33:39.751Z
120663,13276442322,08/06/2026 �s 15:38:20,08/06/2026 �s 15:38:20,08/06/2026 �s 15:40:18,2026-10-16T21:33:39.751Z
45377,74032831005,23/09/2025 �s 14:41:45,23/09/2025 �s 14:41:45,23/09/2025 �s 14:43:35,2026-10-16T21:33:39.751Z
96072,12791762924,02/04/2026 �s 05:52:42,02/04/2026 �s 05:52:42,02/04/2026 �s 05:55:01,2026-10-16T21:33:39.751Z
90851,18601632897,17/03/2026 �s 09:04:43,17/03/2026 �s 09:04:43,17/03/2026 �s 09:06:41,2026-10-16T21:33:39.751Z
100413,68824391895,18/04/2026 �s 13:24:43,18/04/2026 �s 13:26:20,18/04/2026 �s 13:45:47,2026-10-16T21:33:39.751Z
41450,65736093750,03/08/2025 �s 08:44:59,03/08/2025 �s 09:14:42,03/08/2025 �s 10:12:10,2026-10-16T21:33:39.751Z
122039,53817300416,24/06/2026 �s 19:47:08,24/06/2026 �s 19:47:08,24/06/2026 �s 19:51:18,2026-10-16T21:33:39.751Z



--- tb_clientes ---


...

id_canal,nome_canal,tipo_canal,status_canal,ingestion_datetime
1,URA,Autosservico,ativo,2026-10-16T22:08:20.017Z
2,CHATBOT,Autosservico,ativo,2026-10-16T22:08:20.017Z
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-16T22:08:20.017Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-16T22:08:20.017Z
1,URA,Autosservico,ativo,2026-10-16T21:37:15.054Z
2,CHATBOT,Autosservico,ativo,2026-10-16T21:37:15.054Z
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-16T21:37:15.054Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-16T21:37:15.054Z
1,URA,Autosservico,ativo,2026-10-16T21:33:59.735Z
2,CHATBOT,Autosservico,ativo,2026-10-16T21:33:59.735Z



--- tb_base_motivos ---


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartão,Cartao,Alta,2026-10-16T21:35:32.391Z
7,Compra não autorizada,Cartao,alta,2026-10-16T21:35:32.391Z
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-16T21:35:32.391Z
14,Transferência de agência,null,Media,2026-10-16T21:35:32.391Z
9,Problema com aplicativo,Atendimento,Media,2026-10-16T21:35:32.391Z
8,Consulta de contrato,Financeiro,Media,2026-10-16T21:35:32.391Z
15,Renegociação de dívida,Financeiro,alta,2026-10-16T21:35:32.391Z
16,Cancelamento de conta,null,Alta,2026-10-16T21:35:32.391Z
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,2026-10-16T21:35:32.391Z
3,Contestação de fatura,Financeiro,alta,2026-10-16T21:35:32.391Z



--- tb_base_atendentes ---


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela França,1,2019-04-18,2025-11-16,inativo,2026-10-16T22:08:25.602Z
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo,2026-10-16T22:08:25.602Z
3,William Marques,1,2019-05-28,2025-04-15,inativo,2026-10-16T22:08:25.602Z
4,Solange Silveira,Nível 1,2019-10-07,null,ativo,2026-10-16T22:08:25.602Z
5,Joaquim Barros,Nível 1,2020-12-06,2026-05-23,inativo,2026-10-16T22:08:25.602Z
6,Roberta Assunção,1,2021-10-29,null,ativo,2026-10-16T22:08:25.602Z
7,Luiza Galvão,Nível 1,2022-04-24,null,ativo,2026-10-16T22:08:25.602Z
8,Josefa Brito,Nível 1,2023-01-25,null,ativo,2026-10-16T22:08:25.602Z
9,Jaqueline Campos,1,2019-04-19,null,ativo,2026-10-16T22:08:25.602Z
10,Marcos Torres,1,2020-01-20,null,ativo,2026-10-16T22:08:25.602Z
